<a href="https://colab.research.google.com/github/shubham-senani/Marketing-Analytics/blob/main/MA12_Product_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Product Analytics

## Loading Data

In [ ]:
# ==========================================
# INSTALL REQUIRED LIBRARIES (Colab only)
# ==========================================
!pip install statsmodels --quiet

# ==========================================
# IMPORT LIBRARIES
# ==========================================
import pandas as pd
import numpy as np
import re
import io
from google.colab import files

import statsmodels.api as sm
from statsmodels.discrete.conditional_models import ConditionalLogit

In [ ]:
def load_dataset(google_sheet_id=None):
    """
    Loads dataset either by:
    1 → Uploading a CSV file manually
    2 → Loading a Google Sheet (if ID is provided)

    Parameters:
    google_sheet_id (str, optional): Google Sheet ID.
                                     If provided, automatically loads Sheet.

    Returns:
    pd.DataFrame
    """

    # -----------------------------------
    # IF GOOGLE SHEET ID IS PROVIDED → AUTO LOAD (Option 2)
    # -----------------------------------
    if google_sheet_id is not None:
        data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
        df = pd.read_csv(data_url)
        print("Google Sheet loaded successfully.")

    else:
        # Ask user only if ID not provided
        print("Choose how you want to provide the dataset:")
        print("1 → Upload CSV file (from your computer)")
        print("2 → Load Google Sheet (as CSV)")

        choice = input("Enter 1 or 2: ").strip()

        # OPTION 1: Upload CSV
        if choice == "1":
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
            print(f"File '{filename}' uploaded successfully.")

        # OPTION 2: Ask for Google Sheet ID
        elif choice == "2":
            google_sheet_id = input("Enter the Google Sheet ID: ").strip()
            data_url = f"https://docs.google.com/uc?id={google_sheet_id}&export=download"
            df = pd.read_csv(data_url)
            print("Google Sheet loaded successfully.")

        else:
            raise ValueError("Invalid choice. Please rerun and enter 1 or 2.")

    # -----------------------------------
    # DATA CHECK
    # -----------------------------------
    print("\nDataset shape:", df.shape)
    print("Columns:", df.columns.tolist())

    return df

## Discrete Choice Modelling

In [ ]:
# Call the custom function to load the dataset
# The function will prompt you to either:
# 1) Upload a CSV file manually, or
# 2) Enter a Google Sheet ID to load data directly
# 1eNiX7vUTZc8wKUG9ZgPujxiKUAvu_1xt
df = load_dataset(google_sheet_id="1eNiX7vUTZc8wKUG9ZgPujxiKUAvu_1xt")

# Display the first 5 rows of the dataset
df.head()

Google Sheet loaded successfully.

Dataset shape: (2798, 14)
Columns: ['id', 'disp.heinz41', 'disp.heinz32', 'disp.heinz28', 'disp.hunts32', 'feat.heinz41', 'feat.heinz32', 'feat.heinz28', 'feat.hunts32', 'price.heinz41', 'price.heinz32', 'price.heinz28', 'price.hunts32', 'choice']


,id,disp.heinz41,disp.heinz32,disp.heinz28,disp.hunts32,feat.heinz41,feat.heinz32,feat.heinz28,feat.hunts32,price.heinz41,price.heinz32,price.heinz28,price.hunts32,choice
0,1,0,0,0,0,0,0,0,0,4.6,3.7,5.2,3.4,heinz28
1,1,0,0,0,0,0,0,0,0,4.6,4.3,5.2,4.4,heinz28
2,1,0,0,0,0,0,1,0,0,4.6,2.5,4.6,4.8,heinz28
3,1,0,0,0,0,0,0,0,0,4.6,3.7,5.2,3.4,heinz28
4,1,0,0,0,0,0,0,1,0,4.6,3.0,4.6,4.8,heinz28


This dataset contains scanner panel data on **ketchup purchases.**


**Sample Size:** 300 households (unique consumers),
2,798 purchase occasions (choice situations)


**Key Variables:**

*   id → Household (individual) identifier
*   choice → Chosen product alternative:
    *   Heinz 28 oz, Heinz 32 oz, Heinz 41 oz, Hunts 32 oz
*   disp.z → Binary indicator (1/0) if brand z had an in-store display
*   feat.z → Binary indicator (1/0) if brand z had a newspaper feature ad
*   price.z → Price of brand z during that purchase occasion


**Product Structure**

There are four alternatives which represent the most commonly purchased ketchup sizes.Consumers may exhibit size loyalty, meaning substitution across sizes may be limited.

**Analytical Objective**

This dataset is used to analyze consumer demand and brand choice behavior by evaluating:
*   Impact of price reductions on purchase probability
*   Effect of display and feature promotions on sales lift
*   Effectiveness of retailer-supported promotions

**Strategic Insight:**

If promotions significantly increase sales, brands should negotiate with retailers to strengthen in-store advertising support and improve promotional ROI.

In [ ]:
# ==========================================
# WIDE → LONG TRANSFORMATION
# (Equivalent to R: dfidx(..., shape="wide"))
# ==========================================

df = df.copy()

# Create observation ID (one choice situation per row)
df["obs_id"] = df.index

# Collect alternatives from columns like "price.heinz41", "disp.hunts32", etc.
alts = set()
for c in df.columns:
    if "." in c:
        left, right = c.split(".", 1)
        if left in ["price", "disp", "feat"]:
            alts.add(right)

alts = sorted(alts)
print("Detected Alternatives:", alts)

# Safety check: ensure all required columns exist for each alternative
missing = []
for alt in alts:
    for v in ["price", "disp", "feat"]:
        col = f"{v}.{alt}"
        if col not in df.columns:
            missing.append(col)

if missing:
    raise ValueError(f"Missing expected columns: {missing[:10]} ... (showing first 10)")

# Build long format
# One row = one alternative inside one choice situation instead of One row = one shopping occasion
long_df = pd.concat(
    [
        pd.DataFrame({
            "obs_id": df["obs_id"].values,
            "alternative": alt,
            "price": df[f"price.{alt}"].values,
            "disp":  df[f"disp.{alt}"].values,
            "feat":  df[f"feat.{alt}"].values,
            "chosen": (df["choice"].astype(str) == alt).astype(int).values
        })
        for alt in alts
    ],
    ignore_index=True
)

print("Long data shape:", long_df.shape)
long_df.head()

Detected Alternatives: ['heinz28', 'heinz32', 'heinz41', 'hunts32']
Long data shape: (11192, 6)


,obs_id,alternative,price,disp,feat,chosen
0,0,heinz28,5.2,0,0,1
1,1,heinz28,5.2,0,0,1
2,2,heinz28,4.6,0,0,1
3,3,heinz28,5.2,0,0,1
4,4,heinz28,4.6,0,1,1


In [ ]:
# ==========================================
# INTERACTION TERMS
# ==========================================

long_df["price_disp"] = long_df["price"] * long_df["disp"]
long_df["price_feat"] = long_df["price"] * long_df["feat"]

long_df.head()

,obs_id,alternative,price,disp,feat,chosen,price_disp,price_feat
0,0,heinz28,5.2,0,0,1,0.0,0.0
1,1,heinz28,5.2,0,0,1,0.0,0.0
2,2,heinz28,4.6,0,0,1,0.0,0.0
3,3,heinz28,5.2,0,0,1,0.0,0.0
4,4,heinz28,4.6,0,1,1,0.0,4.6


In [ ]:
# ==========================================
# CONDITIONAL LOGIT MODEL (Equivalent to R mlogit)
# ==========================================

# -------------------------------------------------
# Create Alternative-Specific Constants (ASCs)
# -------------------------------------------------
# Each brand gets its own intercept (except one base brand).
# drop_first=True avoids dummy variable trap (one brand becomes reference).
alt_dummies = pd.get_dummies(long_df["alternative"], drop_first=True)


# -------------------------------------------------
# Construct Design Matrix (X matrix)
# -------------------------------------------------
# Combine:
# - Alternative-specific intercepts (brand dummies)
# - Explanatory variables:
#     price
#     display advertising (disp)
#     feature advertising (feat)
# - Interaction terms:
#     price * disp
#     price * feat
X = pd.concat(
    [
        alt_dummies,  # Brand intercepts
        long_df[["price", "disp", "feat", "price_disp", "price_feat"]]
    ],
    axis=1
).astype(float)


# -------------------------------------------------
# Define Dependent Variable (Choice Indicator)
# -------------------------------------------------
# chosen = 1 if that alternative was selected
# chosen = 0 otherwise
y = long_df["chosen"].astype(int)


# -------------------------------------------------
# Define Choice Situation Groups
# -------------------------------------------------
# groups identify each choice occasion (obs_id).
# Conditional Logit compares alternatives *within* each group.
groups = long_df["obs_id"].astype(int)


# -------------------------------------------------
# Estimate Conditional Logit Model
# -------------------------------------------------
# This estimates:
# U_ij = ASC_j + β1*price + β2*disp + β3*feat
#        + β4*(price*disp) + β5*(price*feat)
model = ConditionalLogit(y, X, groups=groups)

# maxiter=200 ensures convergence if dataset is large
result = model.fit(maxiter=200)

In [ ]:
# -------------------------------------------------
# Display Estimation Results
# -------------------------------------------------
# Output includes:
# - Coefficients (Estimates)
# - Standard Errors
# - z-statistics
# - p-values
print(result.summary())

                  Conditional Logit Model Regression Results                  
Dep. Variable:                 chosen   No. Observations:                11192
Model:               ConditionalLogit   No. groups:                       2798
Log-Likelihood:               -2511.9   Min group size:                      4
Method:                          BFGS   Max group size:                      4
Date:                Thu, 26 Feb 2026   Mean group size:                   4.0
Time:                        05:06:06                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
heinz32       -0.9114      0.077    -11.794      0.000      -1.063      -0.760
heinz41       -1.0580      0.088    -12.034      0.000      -1.230      -0.886
hunts32       -2.3971      0.096    -24.844      0.000      -2.586      -2.208
price         -1.4425      0.060    -24.127      0.0

In [ ]:
# ==========================================
# McFADDEN R² (Pseudo R-squared for Logit)
# ==========================================

# -------------------------------------------------
# Estimate NULL (Restricted) Model
# -------------------------------------------------
# The null model contains ONLY alternative-specific constants (ASCs).
# It does NOT include price, display, feature, or interaction terms.
# This represents a baseline model where choice depends only on brand preference.
X_null = alt_dummies.astype(float)

model_null = ConditionalLogit(y, X_null, groups=groups)

# disp=False suppresses iteration output
result_null = model_null.fit(maxiter=200, disp=False)


# -------------------------------------------------
# Extract Log-Likelihood Values
# -------------------------------------------------
# ll_full  = Log-likelihood of FULL model (with all variables)
# ll_null  = Log-likelihood of NULL model (intercepts only)

ll_full = result.llf
ll_null = result_null.llf


# -------------------------------------------------
# Compute McFadden R²
# -------------------------------------------------
# Formula:
# McFadden R² = 1 − (LL_full / LL_null)
#
# Interpretation:
# - Measures improvement of full model over null model
# - Similar idea to R² in OLS (but not the same)
# - Typical good values in choice models: 0.20 – 0.40

mcfadden_r2 = 1 - (ll_full / ll_null)


# -------------------------------------------------
# Print Results
# -------------------------------------------------
print("Log-Likelihood (Full):", ll_full)
print("Log-Likelihood (Null):", ll_null)
print("McFadden R²:", round(mcfadden_r2, 4))

Log-Likelihood (Full): -2511.9104464934335
Log-Likelihood (Null): -3139.038031545968
McFadden R²: 0.1998


## Conjoint Analysis


In [ ]:
# ----------------------------
# ATTRIBUTE LEVELS
# ----------------------------
levels = {
    "Brand": ["Dell", "HP", "Lenovo", "Asus", "Mi", "Others"],           # 6
    "Processor": ["i5", "i7", "Celeron N4020", "Ryzen 3", "Ryzen 5"],    # 5
    "RAM": ["4GB", "6GB", "8GB"],                                       # 3
    "Storage": ["1TB HDD", "256GB SSD", "512GB SSD"],                   # 3
    "Display": ["14", "15.6"],                                          # 2
    "Color": ["Black", "Grey", "Platinum Grey"],                        # 3
    "Price": [28990, 32999, 34999, 36190, 37490, 38499, 40490, 41999, 43249, 44999]  # 10
}

# OS is constant (Windows 10) -> no need in experimental variation
OS_CONSTANT = "Windows 10"

In [ ]:
# ----------------------------
# Helper: dummy-code and score design quality
# (Goal: near-orthogonal => low avg absolute correlation among columns)
# ----------------------------
def design_score(df):
    X = pd.get_dummies(df, drop_first=True).astype(float)
    # remove constant columns if any
    X = X.loc[:, X.std(axis=0) > 0]
    if X.shape[1] < 2:
        return 1e9
    C = np.corrcoef(X.values, rowvar=False)
    # ignore diagonal
    off_diag = C[~np.eye(C.shape[0], dtype=bool)]
    return np.nanmean(np.abs(off_diag))


# ----------------------------
# Build a fractional design by random-search optimization
# ----------------------------
def build_fractional_design(n_profiles=24, n_candidates=8000, n_tries=200, seed=42):
    rng = np.random.default_rng(seed)

    # Create a large candidate set (randomly sampled)
    cand = pd.DataFrame({
        k: rng.choice(v, size=n_candidates, replace=True)
        for k, v in levels.items()
    }).drop_duplicates().reset_index(drop=True)

    best_df, best_score = None, 1e9

    # Randomly try many subsets and keep the one with best balance/orthogonality score
    for _ in range(n_tries):
        subset = cand.sample(n=min(n_profiles, len(cand)), replace=False, random_state=int(rng.integers(1e9)))
        s = design_score(subset)
        if s < best_score:
            best_score, best_df = s, subset.copy()

    best_df = best_df.reset_index(drop=True)
    best_df.insert(1, "OS", OS_CONSTANT)  # add constant OS for completeness in your profiles
    best_df.insert(0, "ProfileID", np.arange(1, len(best_df)+1))
    return best_df, best_score

In [ ]:
# ----------------------------
# Generate design
# ----------------------------
profiles, score = build_fractional_design(n_profiles=24, n_candidates=10000, n_tries=300, seed=7) #24 Profiles

print("Fractional design created.")
print("Avg abs correlation score (lower is better):", round(score, 4))
profiles.head(10)

Fractional design created.
Avg abs correlation score (lower is better): 0.1538


,ProfileID,Brand,OS,Processor,RAM,Storage,Display,Color,Price
0,1,HP,Windows 10,i7,8GB,256GB SSD,14,Platinum Grey,34999
1,2,Asus,Windows 10,Celeron N4020,6GB,256GB SSD,14,Platinum Grey,37490
2,3,Lenovo,Windows 10,Ryzen 3,4GB,256GB SSD,15.6,Grey,41999
3,4,Asus,Windows 10,Celeron N4020,8GB,1TB HDD,15.6,Black,28990
4,5,Dell,Windows 10,Celeron N4020,4GB,1TB HDD,14,Black,37490
5,6,Lenovo,Windows 10,i5,4GB,256GB SSD,14,Grey,43249
6,7,HP,Windows 10,Celeron N4020,8GB,512GB SSD,14,Grey,34999
7,8,Lenovo,Windows 10,Celeron N4020,6GB,1TB HDD,14,Grey,34999
8,9,Dell,Windows 10,Ryzen 5,4GB,512GB SSD,15.6,Black,36190
9,10,Asus,Windows 10,Ryzen 3,6GB,256GB SSD,14,Grey,28990


In [ ]:
import numpy as np
import pandas as pd

def random_rankings(profiles: pd.DataFrame, n_people: int = 20, seed: int = 42):
    """
    Generates purely random ranking data.
    For each participant: assigns a random permutation of ranks 1..N to the N profiles.
    Ensures:
      - no duplicate ranks within a participant (no ties)
      - all ranks 1..N are used
    """
    rng = np.random.default_rng(seed)
    n_profiles = profiles.shape[0]

    if "ProfileID" not in profiles.columns:
        raise ValueError("profiles must contain a 'ProfileID' column")

    profile_ids = profiles["ProfileID"].to_numpy()

    rows = []
    for pid in range(1, n_people + 1):
        ranks = rng.permutation(np.arange(1, n_profiles + 1))  # a permutation => unique 1..N
        rng.shuffle(profile_ids)  # randomize mapping of ranks to profiles

        for prof_id, rank in zip(profile_ids, ranks):
            rows.append({
                "ParticipantID": pid,
                "ProfileID": int(prof_id),
                "Rank": int(rank)
            })

    rankings_long = pd.DataFrame(rows).sort_values(["ParticipantID", "Rank"]).reset_index(drop=True)
    rankings_wide = rankings_long.pivot(index="ParticipantID", columns="ProfileID", values="Rank").reset_index()

    return rankings_long, rankings_wide


In [ ]:
# --- Example usage ---
rankings_long, rankings_wide = random_rankings(profiles, n_people=20, seed=77)

rankings_long.head(12)

,ParticipantID,ProfileID,Rank
0,1,14,1
1,1,3,2
2,1,9,3
3,1,23,4
4,1,8,5
5,1,20,6
6,1,4,7
7,1,10,8
8,1,6,9
9,1,24,10


In [ ]:
# Assume:
# rankings_long = (ParticipantID, ProfileID, Rank)
# profiles = 24-profile design

data = rankings_long.merge(profiles, on="ProfileID")

# Rank 1 = most preferred, highest rank = least preferred.
# Regression needs higher value = higher preference.
# Reverse the scale to create a proper preference score.
max_rank = data["Rank"].max()
data["Preference"] = (max_rank + 1) - data["Rank"]

data.head()

,ParticipantID,ProfileID,Rank,Brand,OS,Processor,RAM,Storage,Display,Color,Price,Preference
0,1,14,1,Dell,Windows 10,i7,4GB,256GB SSD,14,Platinum Grey,28990,24
1,1,3,2,Lenovo,Windows 10,Celeron N4020,4GB,512GB SSD,14,Platinum Grey,32999,23
2,1,9,3,Asus,Windows 10,Celeron N4020,6GB,256GB SSD,14,Platinum Grey,37490,22
3,1,23,4,Dell,Windows 10,Ryzen 5,4GB,512GB SSD,15.6,Black,36190,21
4,1,8,5,Others,Windows 10,i7,6GB,512GB SSD,14,Black,36190,20


In [ ]:
# -----------------------------------------------------------
# Create Dummy Variables for Categorical Attributes
# -----------------------------------------------------------
# Conjoint requires estimating part-worth utilities for each level.
# Since attributes are categorical, we convert them into dummy variables.
# drop_first=True avoids the dummy variable trap (multicollinearity).

X = pd.get_dummies(
    data[["Brand", "Processor", "RAM", "Storage", "Display", "Color"]],
    drop_first=True
)

# Keep Price numeric because it is continuous
# This allows us to estimate price sensitivity directly.
X["Price"] = data["Price"]

# Convert all predictors to numeric (very important for statsmodels)
X = X.astype(float)

# Dependent variable = transformed preference score
y = data["Preference"].astype(float)

# Add constant term (intercept) for regression
X = sm.add_constant(X)

# Estimate part-worth utilities using OLS
# Coefficients represent the utility contribution of each level.
model = sm.OLS(y, X).fit()

# Display regression results
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             Preference   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     1.265
Date:                Thu, 26 Feb 2026   Prob (F-statistic):              0.215
Time:                        05:53:58   Log-Likelihood:                -1599.5
No. Observations:                 480   AIC:                             3233.
Df Residuals:                     463   BIC:                             3304.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  13.3983    

In [ ]:
# -----------------------------------------------------------
# Extract Part-Worth Utilities
# -----------------------------------------------------------
# Regression coefficients represent estimated part-worth utilities.
# Higher coefficient = stronger positive contribution to preference.

partworths = model.params

# Sort from highest to lowest utility contribution
partworths.sort_values(ascending=False)

,0
const,13.398268
Brand_Others,4.312484
Color_Platinum Grey,2.693103
Brand_HP,1.908891
Display_15.6,1.361106
Brand_Dell,1.040489
Color_Grey,0.080464
Price,-0.000013
Storage_256GB SSD,-0.411288
Processor_Ryzen 3,-0.841151


In [ ]:
# -----------------------------------------------------------
# Calculate Attribute Importance
# -----------------------------------------------------------
# Importance is computed as:
# (Range of part-worths within attribute) / (Total range across all attributes)

importance = {}

for attr in ["Brand", "Processor", "RAM", "Storage", "Display", "Color"]:

    # Identify dummy columns related to this attribute
    cols = [c for c in X.columns if attr in c]

    if cols:
        values = partworths[cols]

        # Range = max utility - min utility
        importance[attr] = values.max() - values.min()

# Total range across all attributes
total = sum(importance.values())

# Convert to percentage importance
for k in importance:
    importance[k] = round(100 * importance[k] / total, 2)

importance

{'Brand': 44.42,
 'Processor': 14.81,
 'RAM': 9.3,
 'Storage': 9.54,
 'Display': 0.0,
 'Color': 21.92}

In [ ]:
# -----------------------------------------------------------
# Estimate Total Utility for Each Product Profile
# -----------------------------------------------------------
# Encode profiles using same dummy structure as regression model

profiles_encoded = pd.get_dummies(
    profiles[["Brand", "Processor", "RAM", "Storage", "Display", "Color"]],
    drop_first=True
)

# Add price variable
profiles_encoded["Price"] = profiles["Price"]

# Add intercept
profiles_encoded = sm.add_constant(profiles_encoded)

# Ensure column order matches training matrix (very important)
profiles_encoded = profiles_encoded.reindex(columns=X.columns, fill_value=0)

# Predict total utility using estimated model
profiles["Estimated_Utility"] = model.predict(profiles_encoded)

# Show highest utility products (most preferred)
profiles.sort_values("Estimated_Utility", ascending=False).head()

,ProfileID,Brand,OS,Processor,RAM,Storage,Display,Color,Price,Estimated_Utility
13,5,Others,Windows 10,i5,6GB,1TB HDD,15.6,Grey,41999,16.031332
1,9,Asus,Windows 10,Celeron N4020,6GB,256GB SSD,14,Platinum Grey,37490,14.223789
21,17,Others,Windows 10,Ryzen 3,8GB,256GB SSD,14,Black,32999,13.951727
4,13,Dell,Windows 10,Celeron N4020,4GB,1TB HDD,14,Black,37490,13.951214
10,4,Dell,Windows 10,i5,8GB,256GB SSD,15.6,Platinum Grey,37490,13.910631
